<a href="https://colab.research.google.com/github/maclandrol/cours-ia-med/blob/master/08_nnUNet_Demo_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 08. nnU-Net - Segmentation Médicale Automatique

**Enseignant:** Emmanuel Noutahi, PhD

---

## Objectifs

Dans ce cours pratique, vous allez:
- Comprendre ce qu'est nnU-Net et pourquoi c'est important
- Voir une démonstration de segmentation automatique
- Comparer nnU-Net avec MedSAM (cours 6)
- Comprendre quand utiliser quel outil

## Contexte Médical

### Qu'est-ce que nnU-Net?

**nnU-Net** (no-new-Net) est le **standard actuel** pour la segmentation d'images médicales en recherche. Il est utilisé pour:

- Segmenter automatiquement des organes (foie, reins, cœur, etc.)
- Délimiter des tumeurs avec précision
- Analyser de grandes cohortes de patients
- Planification chirurgicale et radiothérapie

### Pourquoi "no-new-Net"?

Contrairement à d'autres modèles qui proposent de nouvelles architectures, nnU-Net utilise une architecture éprouvée (U-Net) mais se configure **automatiquement** selon vos données.

**Principe:** Pas besoin d'être expert en deep learning - nnU-Net s'adapte automatiquement.

### Différence avec MedSAM (Cours 6)

| Aspect | MedSAM | nnU-Net |
|--------|--------|----------|
| **Usage** | Interactif (vous guidez avec points) | Automatique (aucune intervention) |
| **Formation** | Modèle général pré-entraîné | S'entraîne sur VOS données spécifiques |
| **Temps** | Instantané (quelques secondes) | Nécessite entraînement (heures/jours) |
| **Précision** | Bonne pour exploration | Excellente pour tâche spécifique |
| **Cas d'usage** | Exploration, cas uniques | Production, grandes cohortes |

**En pratique:** MedSAM pour explorer, nnU-Net pour produire.

## Important

Ce cours est une **démonstration** de nnU-Net. L'entraînement complet d'un modèle nnU-Net nécessite:
- Plusieurs heures/jours de calcul
- GPU puissant
- Dataset annoté de qualité

Nous allons utiliser des **modèles pré-entraînés** pour comprendre le concept sans avoir à entraîner.

## 1. Installation et Configuration

**Note:** nnU-Net a des dépendances spécifiques. L'installation peut prendre 2-3 minutes.

In [ ]:
# Installation de nnU-Net et dépendances
# Cela peut prendre quelques minutes
!pip install -q nnunetv2
!pip install -q torch torchvision
!pip install -q SimpleITK nibabel matplotlib numpy pandas scikit-image

print("Installation terminée!")

In [ ]:
# Importations
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import SimpleITK as sitk
from skimage import measure
import warnings
warnings.filterwarnings('ignore')

# Configuration
plt.rcParams['figure.figsize'] = (14, 8)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Dispositif utilisé: {device}")
print(f"PyTorch version: {torch.__version__}")

if device == 'cpu':
    print("\n⚠️  Attention: Pas de GPU détecté.")
    print("   nnU-Net fonctionnera mais sera plus lent.")
    print("   Pour activer le GPU: Exécution → Modifier le type d'exécution → GPU")

## 2. Comprendre l'Architecture U-Net

### Qu'est-ce qu'une U-Net?

U-Net est une architecture de réseau de neurones spécialement conçue pour la segmentation d'images.

**Forme en "U":**

```
Image d'entrée (512×512)
    ↓
[Encodeur - Descente]
    ↓ 256×256 (détection de features)
    ↓ 128×128 (features plus abstraites)
    ↓ 64×64   (représentation compacte)
    ↓
[Bottleneck - Centre du U]
    32×32     (features les plus abstraites)
    ↓
[Décodeur - Montée]
    ↑ 64×64   (reconstruction + features encodeur)
    ↑ 128×128 (ajout de détails)
    ↑ 256×256 (précision spatiale)
    ↑
Segmentation (512×512)
```

**Principe clé:** Les "skip connections" (connexions sautantes) relient l'encodeur au décodeur, préservant les détails fins.

### Pourquoi U-Net pour la Médecine?

1. **Précision spatiale:** Préserve les détails fins (contours précis)
2. **Fonctionne avec peu de données:** Important car les données médicales annotées sont rares
3. **Architecture éprouvée:** Utilisée depuis 2015, très fiable

### nnU-Net = U-Net + Automatisation

nnU-Net ne change pas l'architecture U-Net, mais:
- Configure automatiquement tous les hyperparamètres
- Adapte la profondeur du réseau selon les données
- Choisit la meilleure stratégie d'augmentation de données
- Optimise le preprocessing

**Résultat:** Performances état de l'art sans expertise en deep learning.

## 3. Démonstration avec un Exemple Simple

Pour cette démonstration, nous allons créer une image médicale simulée et montrer comment nnU-Net la segmenterait.

**Note:** Dans un cas réel, vous utiliseriez des images CT/IRM et un modèle pré-entraîné du Model Zoo de nnU-Net.

In [ ]:
# Création d'une image médicale simulée (coupe abdominale)
def creer_image_abdominale_simulee(size=256):
    """
    Crée une coupe abdominale simulée avec:
    - Foie
    - Reins (gauche et droit)
    - Rate
    """
    # Image de base (intensités tissulaires)
    image = np.random.randn(size, size) * 10 + 50
    
    # Masques des organes
    masque_foie = np.zeros((size, size), dtype=bool)
    masque_rein_droit = np.zeros((size, size), dtype=bool)
    masque_rein_gauche = np.zeros((size, size), dtype=bool)
    masque_rate = np.zeros((size, size), dtype=bool)
    
    # Foie (grand organe, partie supérieure droite)
    y, x = np.ogrid[:size, :size]
    masque_foie = ((x - 180)**2 / 80**2 + (y - 80)**2 / 60**2) < 1
    
    # Rein droit
    masque_rein_droit = ((x - 200)**2 / 20**2 + (y - 170)**2 / 35**2) < 1
    
    # Rein gauche
    masque_rein_gauche = ((x - 80)**2 / 20**2 + (y - 170)**2 / 35**2) < 1
    
    # Rate
    masque_rate = ((x - 60)**2 / 25**2 + (y - 90)**2 / 30**2) < 1
    
    # Appliquer les intensités caractéristiques de chaque organe
    image[masque_foie] = 120 + np.random.randn(np.sum(masque_foie)) * 5
    image[masque_rein_droit] = 90 + np.random.randn(np.sum(masque_rein_droit)) * 5
    image[masque_rein_gauche] = 90 + np.random.randn(np.sum(masque_rein_gauche)) * 5
    image[masque_rate] = 100 + np.random.randn(np.sum(masque_rate)) * 5
    
    # Créer le masque de segmentation combiné
    segmentation = np.zeros((size, size), dtype=np.uint8)
    segmentation[masque_foie] = 1      # Foie = label 1
    segmentation[masque_rein_droit] = 2   # Rein droit = label 2
    segmentation[masque_rein_gauche] = 3  # Rein gauche = label 3
    segmentation[masque_rate] = 4      # Rate = label 4
    
    return image, segmentation

# Créer l'image
image_ct, segmentation_ground_truth = creer_image_abdominale_simulee()

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Image CT simulée
axes[0].imshow(image_ct, cmap='gray')
axes[0].set_title("Image CT Abdominale (Simulée)", fontsize=12, fontweight='bold')
axes[0].axis('off')

# Segmentation "ground truth" (vérité terrain)
cmap = plt.cm.get_cmap('tab10')
axes[1].imshow(segmentation_ground_truth, cmap=cmap, vmin=0, vmax=4)
axes[1].set_title("Segmentation de Référence (Ground Truth)", fontsize=12, fontweight='bold')
axes[1].axis('off')

# Légende
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=cmap(1), label='Foie'),
    Patch(facecolor=cmap(2), label='Rein droit'),
    Patch(facecolor=cmap(3), label='Rein gauche'),
    Patch(facecolor=cmap(4), label='Rate')
]
axes[1].legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.show()

print("Image créée avec succès!")
print(f"Dimensions: {image_ct.shape}")
print(f"Organes segmentés: Foie, Reins (×2), Rate")

## 4. Le Processus nnU-Net (Conceptuel)

Voici ce que nnU-Net ferait avec cette image dans un cas réel:

### Étape 1: Preprocessing Automatique

```python
# nnU-Net analyse automatiquement les données et:
# 1. Détecte la résolution spatiale (espacement des pixels)
# 2. Normalise les intensités
# 3. Recadre l'image pour enlever l'arrière-plan
# 4. Redimensionne si nécessaire
```

**Avantage:** Vous n'avez pas à configurer cela manuellement.

### Étape 2: Configuration Automatique

```python
# nnU-Net décide automatiquement:
# - Profondeur du réseau U-Net (combien de niveaux)
# - Taille des patchs à utiliser
# - Taille du batch
# - Stratégie d'augmentation de données
```

**Basé sur:** Taille des images, mémoire GPU disponible, nombre de classes.

### Étape 3: Entraînement

```python
# nnU-Net s'entraîne avec:
# - Cross-validation 5-fold (pour robustesse)
# - Augmentation de données adaptative
# - Early stopping (arrêt si pas d'amélioration)
# 
# Durée typique: 12-48h selon la taille du dataset
```

### Étape 4: Inférence

Une fois entraîné, nnU-Net segmente de nouvelles images automatiquement.

## 5. Simulation d'une Prédiction nnU-Net

Pour illustrer le concept, simulons ce que nnU-Net produirait:

In [ ]:
def simuler_prediction_nnunet(image, ground_truth, noise_level=0.1):
    """
    Simule une prédiction nnU-Net en ajoutant un peu de bruit
    à la vérité terrain pour montrer que ce n'est pas parfait.
    
    Dans la réalité, nnU-Net prédirait à partir de l'image seule.
    """
    prediction = ground_truth.copy().astype(float)
    
    # Ajouter des erreurs simulées
    # 1. Érosion/dilatation aléatoire des contours
    from scipy import ndimage
    for label in [1, 2, 3, 4]:
        mask = (ground_truth == label)
        if np.random.rand() > 0.5:
            # Érosion légère
            mask = ndimage.binary_erosion(mask, iterations=1)
        else:
            # Dilatation légère
            mask = ndimage.binary_dilation(mask, iterations=1)
        
        prediction[mask] = label
    
    # 2. Quelques pixels mal classés
    num_errors = int(noise_level * image.size)
    error_positions = np.random.randint(0, image.shape[0], (num_errors, 2))
    for pos in error_positions:
        if ground_truth[pos[0], pos[1]] > 0:
            # Changer aléatoirement le label
            other_labels = [l for l in [1, 2, 3, 4] if l != ground_truth[pos[0], pos[1]]]
            prediction[pos[0], pos[1]] = np.random.choice(other_labels)
    
    return prediction.astype(np.uint8)

# Simuler la prédiction
prediction_nnunet = simuler_prediction_nnunet(image_ct, segmentation_ground_truth, noise_level=0.05)

# Visualisation comparative
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Image originale
axes[0].imshow(image_ct, cmap='gray')
axes[0].set_title("Image CT", fontsize=12, fontweight='bold')
axes[0].axis('off')

# Vérité terrain
cmap = plt.cm.get_cmap('tab10')
axes[1].imshow(segmentation_ground_truth, cmap=cmap, vmin=0, vmax=4)
axes[1].set_title("Segmentation Manuelle (Ground Truth)", fontsize=12, fontweight='bold')
axes[1].axis('off')

# Prédiction nnU-Net simulée
axes[2].imshow(prediction_nnunet, cmap=cmap, vmin=0, vmax=4)
axes[2].set_title("Prédiction nnU-Net (Simulée)", fontsize=12, fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.show()

# Calcul de métriques
from sklearn.metrics import accuracy_score, jaccard_score

# Aplatir les arrays pour calculer les métriques
gt_flat = segmentation_ground_truth.flatten()
pred_flat = prediction_nnunet.flatten()

# Précision globale
accuracy = accuracy_score(gt_flat, pred_flat)

# Dice score par organe
print("\nMÉTRIQUES DE PERFORMANCE (Simulées):")
print("="*50)
print(f"Précision globale: {accuracy*100:.1f}%")
print("\nDice Score par organe:")

organes = {1: 'Foie', 2: 'Rein droit', 3: 'Rein gauche', 4: 'Rate'}
for label, nom in organes.items():
    # Calculer le Dice score pour cet organe
    gt_organ = (gt_flat == label)
    pred_organ = (pred_flat == label)
    
    intersection = np.sum(gt_organ & pred_organ)
    dice = 2 * intersection / (np.sum(gt_organ) + np.sum(pred_organ))
    
    print(f"  {nom:15s}: {dice*100:.1f}%")

print("="*50)
print("\nNote: Ces résultats sont simulés pour illustration.")
print("Un vrai nnU-Net entraîné atteint typiquement 85-95% de Dice score.")

## 6. Applications Cliniques de nnU-Net

### Oncologie

**Segmentation de tumeurs:**
- Tumeurs cérébrales (gliomes)
- Tumeurs hépatiques
- Métastases pulmonaires

**Avantages:**
- Mesure volumétrique précise
- Suivi de l'évolution sous traitement
- Planification de radiothérapie

### Radiologie

**Segmentation d'organes:**
- Cœur et grandes vaisseaux
- Foie, rate, reins
- Segmentation osseuse

**Avantages:**
- Automatisation du workflow
- Reproductibilité
- Gain de temps (5 min → 30 secondes)

### Neurologie

**Segmentation cérébrale:**
- Matière grise/blanche
- Structures sous-corticales
- Lésions de la SEP

**Avantages:**
- Volumétrie précise
- Détection de changements subtils
- Aide au diagnostic différentiel

### Recherche Clinique

**Analyses de cohortes:**
- Segmentation de 1000+ patients
- Standardisation des mesures
- Études multi-centriques

**Exemple concret:** Le Medical Segmentation Decathlon
- 10 tâches de segmentation différentes
- nnU-Net a gagné ou était dans le top 3 pour toutes
- Sans modification de code entre les tâches!

## 7. Comparaison: MedSAM vs nnU-Net

### Quand Utiliser MedSAM? (Cours 6)

✅ **Bon pour:**
- Exploration rapide
- Cas uniques ou rares
- Pas de données d'entraînement disponibles
- Segmentation interactive souhaitée
- Structures variées

❌ **Moins bon pour:**
- Production à grande échelle
- Besoin de reproductibilité parfaite
- Automatisation complète

**Exemple d'usage:** Segmenter rapidement une tumeur inhabituelle pour planifier une biopsie.

### Quand Utiliser nnU-Net?

✅ **Bon pour:**
- Production à grande échelle
- Tâche de segmentation bien définie
- Dataset annoté disponible (≥50 cas)
- Besoin de haute précision
- Automatisation complète

❌ **Moins bon pour:**
- Cas uniques
- Pas de données d'entraînement
- Prototypage rapide
- Besoin de flexibilité

**Exemple d'usage:** Segmenter automatiquement le foie de 500 patients pour une étude de recherche.

### Approche Hybride (Idéale)

1. **Phase d'exploration:** Utiliser MedSAM pour annoter quelques cas et tester la faisabilité
2. **Annotation:** Créer un dataset (50-200 cas) avec MedSAM ou annotation manuelle
3. **Production:** Entraîner nnU-Net sur ce dataset
4. **Déploiement:** Utiliser nnU-Net pour segmenter automatiquement de nouveaux cas
5. **Cas difficiles:** Revenir à MedSAM pour les cas où nnU-Net échoue

## 8. Limitations et Considérations Pratiques

### Limitations de nnU-Net

1. **Données nécessaires:**
   - Minimum: ~50 cas annotés
   - Idéal: 100-500 cas
   - L'annotation prend du temps (1-2h par cas selon la complexité)

2. **Resources computationnelles:**
   - GPU nécessaire (entraînement)
   - Temps: 12-48h d'entraînement
   - Stockage: Plusieurs GB selon le dataset

3. **Généralisation:**
   - Performe bien sur des données similaires à l'entraînement
   - Peut échouer sur données très différentes (autre scanner, protocole, etc.)
   - Nécessite re-entraînement si changement majeur

4. **Boîte noire:**
   - Difficile d'expliquer pourquoi une erreur
   - Pas d'incertitude quantifiée par défaut
   - Nécessite validation clinique

### Bonnes Pratiques

**Avant d'utiliser nnU-Net:**

1. ✅ Vérifier qu'un modèle pré-entraîné n'existe pas déjà (Model Zoo)
2. ✅ Évaluer si vous avez assez de données annotées
3. ✅ Définir clairement la tâche de segmentation
4. ✅ Planifier les resources (GPU, temps)

**Pendant l'utilisation:**

1. ✅ Toujours valider sur un set de test indépendant
2. ✅ Vérifier manuellement un échantillon de prédictions
3. ✅ Documenter la version, les paramètres, le dataset
4. ✅ Monitorer la performance en production

**Validation clinique:**

1. ✅ Comparer avec segmentation manuelle (gold standard)
2. ✅ Évaluer sur différents sous-groupes (âge, pathologie, scanner)
3. ✅ Identifier les cas d'échec
4. ✅ Définir un seuil de qualité acceptable
5. ✅ Obtenir approbation éthique si recherche clinique

## 9. Résumé et Points Clés

### Ce que vous avez appris:

1. **nnU-Net est l'état de l'art** pour la segmentation médicale automatique
2. **Architecture U-Net** avec configuration automatique
3. **Différence MedSAM vs nnU-Net:** Interactif vs Automatique
4. **Applications cliniques:** Oncologie, radiologie, neurologie, recherche
5. **Limitations:** Données, GPU, temps, généralisation

### Compétences acquises:

- Comprendre quand utiliser nnU-Net vs MedSAM
- Connaître le workflow nnU-Net (preprocessing → entraînement → inférence)
- Évaluer la faisabilité d'un projet nnU-Net
- Comprendre les métriques (Dice score, précision)

### Pour Aller Plus Loin

**Si vous voulez utiliser nnU-Net pour votre recherche:**

1. **Documentation officielle:** https://github.com/MIC-DKFZ/nnUNet
2. **Model Zoo:** Modèles pré-entraînés disponibles
3. **Tutoriels:** Installation, préparation données, entraînement
4. **Forum:** Support communautaire actif

**Datasets publics pour s'entraîner:**

- Medical Segmentation Decathlon
- BTCV (Multi-organ segmentation)
- BraTS (Tumeurs cérébrales)
- KiTS (Tumeurs rénales)

### Message Final

nnU-Net n'est **pas magique** - il nécessite:
- Des données de qualité
- Du temps et des resources
- Une validation rigoureuse

Mais quand ces conditions sont réunies, c'est **l'outil le plus fiable** actuellement pour la segmentation médicale automatique.

**Utilisez-le de manière responsable et validez toujours cliniquement!**

## 10. Questions de Réflexion

1. **Workflow clinique:** Comment intégreriez-vous nnU-Net dans un service de radiologie? Quels seraient les freins et facilitateurs?

2. **Validation:** Combien de cas devriez-vous vérifier manuellement avant de faire confiance aux prédictions nnU-Net en production?

3. **Choix d'outil:** Vous devez segmenter des tumeurs pulmonaires. Vous avez 10 cas à faire cette semaine et 500 dans les 6 prochains mois. Quelle stratégie adoptez-vous?

4. **Responsabilité:** Si nnU-Net sous-estime la taille d'une tumeur et que cela affecte le plan de traitement, qui est responsable?

5. **Généralisation:** Votre nnU-Net est entraîné sur des CT d'un scanner Siemens. Peut-on l'utiliser sur des CT d'un scanner GE? Quelles vérifications faire?